# Описание эксперимента моделирования динамики временных рядов с использованием хаотических нейронных сетей


## Схема пайплайна
Красивый рисунок со схемой будет готов чуть позже


```text
┌──────────────┐
│   Данные     │
│ OHLCV, тикеры│
└──────┬───────┘
       │
       v
┌──────────────────────┐
│ Предобработка данных,│  
│ лог-доходности       │
└──────┬───────────────┘
       │
       ├───────────────────────────────┐
       │                               │
       v                               v
┌──────────────────────┐      ┌──────────────────────────┐
│ Извлечение признаков │      │ Прогнозирование          │
│ (хвосты, память,     │      │ Фиксация метрик качества │
│ сложность, хаос)     │      │ прогнозирования          │
└──────┬───────────────┘      └───────────┬──────────────┘
       │                                  │
       v                                  │
┌──────────────────────────┐              │
│ Отбор и консолидация     │              │
│ финальных наборов        │              │
│ признаков                │              │
└──────┬───────────────────┘              │
       │                                  │
       |                                  │
       │                                  │
       v                                  │
┌──────────────────────┐                  │
│ Кластеризация        │                  │
│ и профилирование     │                  │
│ кластеров            │                  │
└──────┬───────────────┘                  │
       │                                  │
       │                                  │
       │                                  │
       │                                  │
       │                                  │
       v                                  v
┌──────────────────────┐        ┌────────────────────────┐
│ Аналитический этап   │───────>│ Метамоделирование      │
│                      │        │ выбор лучшей модели    │
│                      │        │ по свойствам ряда      │
└──────────────────────┘        └────────────────────────┘
```
Кластеризация в данном проекте используется как аналитический вспомогательный этап: она помогает выявлять режимы рядов и интерпретировать поведение моделей, но не является основным механизмом выбора модели. Основной итоговый механизм выбора реализован через метамоделирование.

## 2. Цели исследования
1. Анализ применимости хаотических нейронных сетей для прогнозирования финансовых временных рядов; определить, где они могут работать лучше, включая анализ посредством кластеризации.
2. Изучить, можно ли выбрать лучшую модель прогнозирования из свойств временных рядов с помощью метамоделирования (построения модели, предсказывающей лучшую модель для прогнозирования).


## 3. Структура пайплайна и репозитория

Проект организован как воспроизводимый исследовательский пайплайн, в котором отдельные этапы последовательно преобразуют исходные рыночные данные в признаки, результаты прогнозирования и итоговые артефакты метамоделирования.

Основные стадии пайплайна:

1. **Подготовка данных.**  
   Формируются каталог рядов, профили датасета и таблица лог-доходностей. На этом этапе задаётся состав исследуемой выборки и базовая структура временных рядов.

2. **Построение и отбор признаков.**  
   Для каждого ряда вычисляются группы признаков, описывающие статистические свойства, временную зависимость, структурную сложность и хаотические характеристики. Затем проводится скрининг, консолидация и сборка финальных наборов признаков.

3. **Кластеризация и профилирование кластеров.**  
   На основе финальных признаков строятся кластеризации рядов, после чего выполняется профилирование кластеров и анализ того, как различаются режимы рядов и качество моделей в этих режимах. Этот этап используется как аналитический, а не как основной механизм выбора модели.

4. **Сравнение моделей прогнозирования.**  
   На выбранной выборке запускается сравнение набора моделей прогнозирования на нескольких горизонтах. Результатом этапа становятся метрики качества по моделям, горизонтам и рядам.

5. **Метамоделирование.**  
   На финальном этапе строится модель, которая по свойствам ряда пытается выбрать наиболее подходящую модель прогнозирования. Именно этот этап является основным итоговым результатом проекта.

Структурно репозиторий разделён на несколько основных блоков:
- `src/` — реализация этапов пайплайна;
- `configs/` — конфигурации запусков;
- `artifacts/` — выходные артефакты, таблицы, отчёты;
- `docs/` — поясняющие материалы;
- `tests/` — проверка корректности ключевых этапов.

С точки зрения научного содержания центральная логика проекта выглядит так: сначала исследуются свойства рядов и поведение различных моделей прогнозирования, затем проверяется, можно ли использовать эти свойства для выбора наилучшей модели прогнозирования с помощью метамоделирования.


## 4. Данные и предварительная обработка

### 4.1. Состав исходных данных

Исходные данные проекта представляют собой набор рыночных временных рядов по российским и американским инструментам. На этапе подготовки данных формируются:
- каталог рядов;
- профили датасета;
- таблица лог-доходностей, используемая в дальнейших этапах анализа.

Краткие характеристики подготовленных данных приведены в таблице ниже.

| Показатель | Значение |
|---|---:|
| Число рядов в каталоге | 7444 |
| Число рядов US | 7195 |
| Число рядов RU | 249 |
| Размер профиля `core_balanced` | 418 |
| Размер профиля `extended_us_heavy` | 1009 |

В дальнейшем для основных экспериментов использовался профиль `core_balanced` (сбалансирован по количеству рядов RU и US), обеспечивающий более сбалансированное покрытие рядов по рынкам.

### 4.2. Группы признаков и финальный отбор

На этапе построения признаков в проекте использовались четыре содержательные группы:

1. **Признаки зависимости и памяти** (`block_a_dependence.py`)  
   Описывают долгосрочную память, автокорреляционную структуру и отклонения от случайного блуждания.

2. **Спектральные признаки и признаки сложности** (`block_b_spectrum.py`)  
   Описывают спектральную форму ряда, энтропию, алгоритмическую сложность.

3. **Признаки распределения и хвостов** (`block_c_tails.py`)  
   Описывают тяжесть хвостов, форму распределения и устойчивые характеристики экстремальных значений.

4. **Хаотические и нелинейно-динамические признаки** (`block_d_chaos.py`)  
   Предназначены для описания вложения, размерности, временной задержки и характеристик хаотической динамики.

Изначально пространство признаков было шире, однако после этапов скрининга, консолидации и формирования финальных наборов были оставлены только признаки, вошедшие в финальные таблицы `final_clustering_features_base_v1.parquet` и `final_clustering_features_with_chaos_v1.parquet`. В коде `final_feature_sets.py` зафиксирован следующий итоговый состав.

**Финальный базовый набор признаков (`22` признака):**

- **Зависимость и память**  
  `hurst_rs`, `hurst_dfa`, `acf_lag_2`, `acf_lag_5`, `acf_lag_10`, `acf_lag_25`, `acf_lag_50`, `acf_lag_100`, `abs_acf_lag_2`, `abs_acf_lag_5`, `abs_acf_lag_10`, `abs_acf_lag_25`, `abs_acf_lag_50`, `vr_q10`, `lb_ret_stat_50`

- **Спектральные признаки и признаки сложности**  
  `lz_complexity`, `permutation_entropy`, `spectral_flatness`

- **Признаки распределения и хвостов**  
  `kurtosis`, `robust_kurtosis`, `tail_ratio_upper`, `hill_tail_index`

**Дополнительные хаотические признаки (`+3` к базовому набору):**
- `correlation_dimension`
- `embedding_dimension`
- `selected_delay_tau`

Итоговые артефакты имеют вид:
- `final_clustering_features_base_v1.parquet` — `22` признака;
- `final_clustering_features_with_chaos_v1.parquet` — `25` признаков.

### 4.3. Общая схема разбиения и оценки

Для этапа прогнозирования использовалась схема временного разбиения `rolling origin`. В такой постановке модель обучается только на прошлом участке ряда и проверяется на следующем по времени отрезке; затем точка разделения может сдвигаться вперёд. В текущем режиме использовался один такой временной split (`n_folds = 1`).

## 5. Модели прогнозирования
- baseline: 
    `naive_zero` - прогноз 0, 
    `naive_mean` - прогноз среднего.
- classic: 
    `ridge_lag` - ридж-регрессия,
    `esn` - резервуарная модель,
    `vanilla_mlp` - многослойный перцептрон,
    `lstm_forecast` - LSTM.
- chaotic: 
    `chaotic_esn` 
    `transient_chaotic_esn`, 
    `chaotic_mlp`, 
    `chaotic_lstm_forecast`, 
    `chaotic_logistic_net`

- **`chaotic_esn`**  
  Это вариант `ESN`, в котором хаотический режим задаётся через изменение спектрального радиуса резервуара.

- **`transient_chaotic_esn`**  
  В `TransientChaoticESN` хаотичность задаётся не постоянным параметром, а переходным режимом: рекуррентный вклад масштабируется временным коэффициентом `g_t`, который убывает от `g_start` к `g_end`. То есть модель стартует в более хаотическом режиме и затем постепенно переходит к более стабильной динамике.

- **`chaotic_mlp`**  
  В `ChaoticMLP` используется хаотическая функция активация, построенная на основе logistic map

- **`chaotic_lstm_forecast`**  
  В `LSTMForecastChaotic` архитектура LSTM сохраняется, но веса инициализируются не стандартным способом, а хаотической последовательностью logistic map 

- **`chaotic_logistic_net`**  
  В `ChaoticLogisticNet` скрытая динамика непосредственно строится на логистическом отображении: состояние обновляется по формуле вида `r_t * h * (1 - h)` с демпфированием через `beta`


## 6. Постановка эксперимента по прогнозированию

Основной прогон в текущей версии проекта выполнялся на профиле `core_balanced` и включал прогнозирование на трёх горизонтах: 1, 5 и 20 шагов вперёд. Для каждого горизонта использовался свой размер входного окна

Ключевые параметры постановки приведены ниже.

| Параметр | Значение |
|---|---|
| Профиль данных | `core_balanced` |
| Горизонты прогнозирования | 1, 5, 20 |
| Размеры окон | 64 / 32 / 16 |
| Схема валидации | `rolling_origin` |
| Число временных разбиений | 1 |
| Число рядов в прогоне | 300 |
| Число моделей | 11 |
| Число задач | 9900 |
| Ошибок / таймаутов | 0 / 0 |
| Ограничение обучения для torch-моделей | `max_epochs = 2`, `early_stopping_patience = 1` |


## 7. Кластеризация и профилирование кластеров
Кластеризация использовалась как аналитический этап для изучения структурной неоднородности и моделирования поведения кластеров.
Этот этап не является окончательным механизмом выбора модели; окончательный выбор осуществляется посредством метамоделирования.
| Параметры эксперимента | Значение |
|---|---|
| `feature_set`(набор признаков) | `base` (без хаотических метрик), `with_chaos` (дополненный хаотическими метриками) |
| `scaler` | `identity`, `robust`, `quantile` |
| `space_type`(тип пространства признаков) | `original`, `pca` |
| `algorithm` | `gmm`, `agglomerative` |
| `k`(количество кластров) | 2..8 |
| `n_bootstrap` | 30 |

После выбора конфигурации выполнялось профилирование кластеров:
- анализ распределения признаков по кластерам;
- выделение ключевых различающих признаков;
- проверка баланса кластеров;
- сопоставление кластеров с качеством моделей прогнозирования.

| Конфигурация | silhouette | davies_bouldin | calinski_harabasz | cluster_min | cluster_max |
|---|---:|---:|---:|---:|---:|
| `cfg_0448` | 0.357561 | 0.822157 | 357.881022 | 10 | 101 |
| `cfg_0185` | 0.424362 | 0.829961 | 278.949190 | 44 | 183 |
| `cfg_0443` | 0.416587 | 0.799226 | 336.956031 | 77 | 201 |
**Источник таблицы:** `artifacts/reports/cluster_profiles_v1.xlsx`

Поэтому роль кластеризации в проекте можно описать так: она не дала готового правила выбора модели, но позволила:
- выделить режимы рядов;
- проверить, насколько различается качество моделей между этими режимами;
- использовать кластерную структуру как дополнительный инструмент интерпретации, в том числе при анализе применимости хаотических моделей.

**[Место для изображения 1: тепловая карта профилей кластеров для конфигурации без хаотических признаков]**  
**Файл:** `artifacts/figures/clusters/cluster_profile_heatmap_cfg_0185_base_gmm_quantile_pca_pca2_k4.png`

**[Место для изображения 2: scatter-визуализация кластеров для конфигурации без хаотических признаков]**  
**Файл:** `artifacts/figures/clusters/cluster_scatter_cfg_0185_base_gmm_quantile_pca_pca2_k4.png`

**[Место для изображения 3: тепловая карта профилей кластеров для конфигурации с хаотическими признаками]**  
**Файл:** `artifacts/figures/clusters/cluster_profile_heatmap_cfg_0448_with_chaos_agglomerative_quantile_pca_pca2_k8.png`

**[Место для изображения 4: scatter-визуализация кластеров для конфигурации с хаотическими признаками]**  
**Файл:** `artifacts/figures/clusters/cluster_scatter_cfg_0448_with_chaos_agglomerative_quantile_pca_pca2_k8.png`


!artifacts/figures/clusters/cluster_profile_heatmap_cfg_0185_base_gmm_quantile_pca_pca2_k4.png

In [2]:
!artifacts/figures/clusters/cluster_profile_heatmap_cfg_0185_base_gmm_quantile_pca_pca2_k4.png

"artifacts" �� ���� ����७��� ��� ���譥�
��������, �ᯮ��塞�� �ணࠬ��� ��� ������ 䠩���.


In [3]:
<img src="artifacts/figures/clusters/cluster_profile_heatmap_cfg_0185_base_gmm_quantile_pca_pca2_k4.png" width="600">

SyntaxError: invalid syntax (208956610.py, line 1)

<img src="artifacts/figures/clusters/cluster_profile_heatmap_cfg_0185_base_gmm_quantile_pca_pca2_k4.png" width="600">

## 8. Результаты прогнозирования

### 8.1. Средние значения RMSE по моделям и горизонтам

В таблице ниже приведены средние значения `RMSE` по каждому сочетанию «модель — горизонт прогнозирования». Для расчёта использовались только данные последнего прогона `forecasting_benchmark_smoke_v1_20260331T201416Z`.

| Модель | h = 1 | h = 5 | h = 20 |
|---|---:|---:|---:|
| `naive_zero` | 0.023440 | 0.068080 | 0.137221 |
| `naive_mean` | 0.023444 | 0.068109 | 0.137401 |
| `ridge_lag` | 0.020826 | 0.068853 | 0.138720 |
| `esn` | 0.017889 | 0.068154 | 0.138989 |
| `vanilla_mlp` | 0.027794 | 0.071036 | 0.139500 |
| `lstm_forecast` | 0.023154 | 0.068214 | 0.137648 |
| `chaotic_esn` | 0.021638 | 0.074447 | 0.145888 |
| `transient_chaotic_esn` | 0.018126 | 0.068614 | 0.141416 |
| `chaotic_mlp` | 0.036129 | 0.070588 | 0.141214 |
| `chaotic_lstm_forecast` | 0.023445 | 0.068296 | 0.137497 |
| `chaotic_logistic_net` | 0.064764 | 0.093023 | 0.152902 |

**Источник таблицы:** `forecasting_smoke_summary_v1.xlsx`, лист `series_metrics_full`; использован только `run_id = forecasting_benchmark_smoke_v1_20260331T201416Z`.

По этой таблице видно, что:
- на горизонте `h = 1` лучшей по `RMSE` является модель `esn`;
- на горизонтах `h = 5` и `h = 20` минимальное среднее значение `RMSE` показывает `naive_zero`;
- `transient_chaotic_esn` на коротком горизонте близка к `esn`, но в среднем уступает ей;
- `chaotic_esn` и `chaotic_logistic_net` в текущем запуске выглядят заметно слабее по `RMSE`, чем базовый `esn`.

**[Место для изображения: тепловая карта средних значений RMSE по моделям и горизонтам]**

### 8.2. Wins-анализ по RMSE

Наряду со средними значениями метрик полезно рассмотреть, сколько раз каждая модель оказывалась лучшей на отдельном ряду. В таблице ниже под победой понимается случай, когда модель показала минимальное значение `RMSE` среди всех кандидатов на данном ряду и данном горизонте.

| Модель | h = 1 | h = 5 | h = 20 |
|---|---:|---:|---:|
| `naive_zero` | 0 | 11 | 23 |
| `naive_mean` | 0 | 14 | 20 |
| `ridge_lag` | 0 | 1 | 1 |
| `esn` | 100 | 51 | 7 |
| `vanilla_mlp` | 0 | 1 | 7 |
| `lstm_forecast` | 0 | 10 | 7 |
| `chaotic_esn` | 0 | 0 | 5 |
| `transient_chaotic_esn` | 0 | 3 | 2 |
| `chaotic_mlp` | 0 | 0 | 6 |
| `chaotic_lstm_forecast` | 0 | 9 | 18 |
| `chaotic_logistic_net` | 0 | 0 | 4 |

**Источник таблицы:** `forecasting_smoke_summary_v1.xlsx`, лист `series_metrics_full`; использован только `run_id = forecasting_benchmark_smoke_v1_20260331T201416Z`. Для каждого сочетания `(series_id, horizon)` выбиралась модель с минимальным `rmse_mean`.

Эта таблица даёт более содержательную картину, чем простое сравнение средних:
- на горизонте `h = 1` модель `esn` доминирует полностью и выигрывает у всех остальных моделей на **100 из 300 рядов**;
- на горизонте `h = 5` лидерство `esn` сохраняется, но уже не является абсолютным: заметную долю побед получают также `naive_mean`, `naive_zero`, `lstm_forecast` и `chaotic_lstm_forecast`;
- на горизонте `h = 20` структура становится значительно более смешанной: чаще всего выигрывают `naive_zero`, `naive_mean` и `chaotic_lstm_forecast`, тогда как `esn` побеждает существенно реже.

Таким образом, по мере увеличения горизонта задача становится менее вырожденной, а распределение лучших моделей по рядам — более разнообразным.

**[Место для изображения: столбчатая диаграмма числа побед моделей по горизонтам]**

### 8.2. Лидеры по горизонтам и метрикам

| Горизонт | Лучшая модель по RMSE | RMSE | Лучшая модель по `directional_accuracy` | `directional_accuracy` |
|---:|---|---:|---|---:|
| 1 | `esn` | 0.025872 | `esn` | 0.672660 |
| 5 | `naive_zero` | 0.094224 | `esn` | 0.529383 |
| 20 | `naive_zero` | 0.182475 | `esn` | 0.509816 |

Эта таблица важна по двум причинам. Во-первых, она показывает, что на коротком горизонте `ESN` лидирует сразу по двум критериям. Во-вторых, на горизонтах 5 и 20 появляется расхождение между метриками: по `RMSE` лучшей становится простая baseline-модель `naive_zero`, тогда как по `directional_accuracy` лидерство сохраняет `ESN`.

**Источник таблицы:** `artifacts/forecasting/series_metrics_smoke_v1.parquet`, `artifacts/reports/forecasting_smoke_summary_v1.xlsx` (сводка по моделям и горизонтам).

### 8.3. Сравнение групп моделей

| Горизонт | Группа моделей | Средний RMSE | Средний `directional_accuracy` |
|---:|---|---:|---:|
| 1 | `baseline` | 0.033638 | 0.260698 |
| 1 | `classic` | 0.031550 | 0.570094 |
| 1 | `chaotic` | 0.040652 | 0.539054 |
| 5 | `baseline` | 0.094249 | 0.252705 |
| 5 | `classic` | 0.095383 | 0.509027 |
| 5 | `chaotic` | 0.101235 | 0.507932 |
| 20 | `baseline` | 0.182671 | 0.257194 |
| 20 | `classic` | 0.188359 | 0.505748 |
| 20 | `chaotic` | 0.199532 | 0.502951 |

Эта агрегация удобна тем, что показывает не только отдельных лидеров, но и общее положение групп моделей. В текущем запуске:
- по `RMSE` baseline-модели конкурентны или даже сильнее на дальних горизонтах;
- по `directional_accuracy` заметное преимущество остаётся за группой `classic`, в которую в данном блокноте включён `ESN`;
- группа `chaotic` не становится универсальным лидером в среднем, но остаётся сопоставимой с `classic` по метрике направления на горизонтах 5 и 20.

**Источник таблицы:** агрегирование по `artifacts/forecasting/series_metrics_smoke_v1.parquet` с группировкой моделей на `baseline / classic / chaotic`.

### 8.4. Распределение побед по группам

| Горизонт | Метрика | Доля побед `baseline` | Доля побед `classic` | Доля побед `chaotic` |
|---:|---|---:|---:|---:|
| 1 | `RMSE` | 0.00 | 0.99 | 0.01 |
| 1 | `directional_accuracy` | 0.00 | 0.94 | 0.06 |
| 5 | `RMSE` | 0.28 | 0.41 | 0.31 |
| 5 | `directional_accuracy` | 0.06 | 0.46 | 0.48 |
| 20 | `RMSE` | 0.36 | 0.33 | 0.31 |
| 20 | `directional_accuracy` | 0.01 | 0.32 | 0.67 |

Эта таблица отвечает уже на другой вопрос: не какая группа лучше в среднем, а как часто модели из данной группы оказываются лучшими на отдельных рядах. Именно здесь видно, что:
- на горизонте 1 задачи почти полностью контролируются группой `classic`;
- на горизонте 5 по `directional_accuracy` уже есть конкуренция между `classic` и `chaotic`;
- на горизонте 20 по `directional_accuracy` на уровне win-count группа `chaotic` выглядит особенно сильной

**Источник таблицы:** `artifacts/meta_modeling/forecasting_model_wins_v1.csv` (`win_count`, нормировка внутри пары `horizon + metric`).

### 8.5. Отдельно о роли ESN

Результаты текущего прогона позволяют выделить `ESN` как ключевую модель проекта. В текущем режиме он:
- доминирует на коротком горизонте;
- сохраняет лидерство по `directional_accuracy` на более дальних горизонтах;
- служит естественным ориентиром как для анализа хаотических модификаций, так и для постановки метамоделирования.

В совокупности таблицы средних значений `RMSE` и wins-анализ показывают, что поведение моделей существенно зависит от горизонта прогнозирования. На коротком горизонте `ESN` выступает как практически доминирующая модель, тогда как на более дальних горизонтах лидерство перераспределяется между baseline-моделями, `LSTM`-подходами и отдельными хаотическими модификациями. Это наблюдение напрямую мотивирует следующий этап работы — метамоделирование, в котором задача выбора модели формулируется явно.

**[Место для изображения 1: сравнительный график RMSE по основным моделям и горизонтам]**  
**Рекомендуемый источник:** `artifacts/reports/forecasting_smoke_summary_v1.xlsx`

**[Место для изображения 2: сравнительный график `directional_accuracy` по основным моделям и горизонтам]**  
**Рекомендуемый источник:** `artifacts/reports/forecasting_smoke_summary_v1.xlsx`

**[Место для изображения 3: отдельный график или таблица для сравнения `ESN`, `chaotic_esn` и `transient_chaotic_esn`]**  
**Рекомендуемые источники:** `artifacts/reports/forecasting_smoke_summary_v1.xlsx`, `artifacts/reports/cluster_forecasting_analysis_v1.xlsx`

## 10. Задача метамоделирования

На заключительном этапе проекта рассматривается задача выбора наилучшей модели прогнозирования по свойствам временного ряда. Для каждой пары `(горизонт, метрика)` формируется отдельная задача: по признакам ряда нужно предсказать, какая из моделей-кандидатов даст наилучший результат.

В проекте использовались две целевые метрики:
- `RMSE` — для оценки масштаба ошибки прогноза;
- `directional_accuracy` — для оценки правильности предсказания направления движения.

Для интерпретации результатов метамоделирования используются три опорные величины:

1. **`best single`** — одна фиксированная модель прогнозирования, которая в среднем показывает лучший результат на данной задаче и затем применяется ко всем рядам без индивидуального выбора.

2. **`oracle`** — недостижимый ориентир, при котором для каждого ряда выбирается фактически лучшая модель уже по известному результату. Эта величина задаёт верхнюю границу качества.

3. **`achieved metric`** — фактическое качество, достигнутое метамоделью, которая выбирает модель прогнозирования по признакам ряда.

Сравнение этих величин позволяет ответить на два вопроса:
- даёт ли метамоделирование практический выигрыш по сравнению с использованием одной глобально лучшей модели;
- насколько далеко текущий результат находится от теоретического предела, задаваемого `oracle`.

При интерпретации улучшения важно учитывать знак метрики:
- для `RMSE` улучшение удобно считать как `best_single - achieved`, то есть положительное значение означает уменьшение ошибки;
- для `directional_accuracy` улучшение считается как `achieved - best_single`, то есть положительное значение означает рост точности.


## 11. Конфигурации метамоделирования
| parameter | values |
|---|---|
| `candidate_set` | `top_3`, `top_4`, `top_5`, `top_6` |
| `feature_set` | `full_features`, `full_plus_basic_features`, `selected_features` |
| `balancing_mode` | `default`, `balanced` |
| decision_rule | `top_1`, `confidence_fallback_0.50/0.60/0.70/0.80` |
| classifiers | `logistic_regression`, `random_forest_classifier`, `catboost_classifier` |
| repeats | 5 |


## 12. Результаты метамоделирования
| horizon | metric | best configuration | achieved | best_single | oracle | improvement | gap_to_oracle |
|---:|---|---|---:|---:|---:|---:|---:|
| 1 | `rmse` | `catboost_classifier`, `top_3`, `full_features`, `balanced`, `confidence_fallback_0.50` | 0.023283 | 0.023283 | 0.023280 | 0.000000 | 0.000003 |
| 1 | `directional_accuracy` | `catboost_classifier`, `top_3`, `full_features`, `balanced`, `confidence_fallback_0.70` | 0.671496 | 0.671496 | 0.671935 | 0.000000 | 0.000439 |
| 5 | `rmse` | `random_forest_classifier`, `top_5`, `full_plus_basic_features`, `default`, `confidence_fallback_0.50` | 0.087472 | 0.087610 | 0.087051 | 0.000138 | 0.000422 |
| 5 | `directional_accuracy` | `catboost_classifier`, `top_4`, `full_plus_basic_features`, `default`, `top_1` | 0.529841 | 0.529576 | 0.533380 | 0.000264 | 0.003540 |
| 20 | `rmse` | `random_forest_classifier`, `top_3`, `full_plus_basic_features`, `balanced`, `confidence_fallback_0.50` | 0.171077 | 0.171110 | 0.170679 | 0.000033 | 0.000398 |
| 20 | `directional_accuracy` | `random_forest_classifier`, `top_6`, `full_plus_basic_features`, `default`, `top_1` | 0.518324 | 0.509125 | 0.535757 | 0.009198 | 0.017434 |
**Источник таблицы:** `artifacts/meta_modeling/best_config_per_task_experiments_v1.csv`, `artifacts/reports/meta_modeling_experiments_v1.xlsx` (`summary`, `comparison`).

| horizon | metric | total_configs | positive_improvement | zero_improvement | mean_majority_class_share |
|---:|---|---:|---:|---:|---:|
| 1 | `rmse` | 360 | 0 | 184 | 0.990 |
| 1 | `directional_accuracy` | 360 | 0 | 177 | 0.887 |
| 5 | `rmse` | 360 | 238 | 31 | 0.397 |
| 5 | `directional_accuracy` | 360 | 15 | 153 | 0.462 |
| 20 | `rmse` | 360 | 70 | 87 | 0.363 |
| 20 | `directional_accuracy` | 360 | 217 | 50 | 0.418 |
Значение mean_majority_class_share: средняя доля наиболее часто встречающегося целевого класса (модель-победитель) в задаче. Более высокие значения указывают на более сильное вырождение задачи выбора.
**Источник таблицы:** `artifacts/meta_modeling/comparison_table_experiments_v1.csv`, `artifacts/meta_modeling/class_distribution_experiments_v1.csv`.


## 13. Интерпретация результатов
- Не существует универсальной модели для всех горизонтов и показателей.
- Метамоделирование дает полезные результаты там, где классовое доминирование умеренно и конкуренция кандидатов реальна.
- Хаотические модели слабы в качестве глобальных средних победителей, но полезны в качестве моделей-кандидатов при анализе маршрутов и режимов.


## 14. Ограничения исследования

Результаты, представленные в данном блокноте, следует интерпретировать с учётом нескольких ограничений.

1. **Ограниченная постановка эксперимента прогнозирования**  
   Основной прогон прогнозирования выполнен с одним временным разбиением и жёстко ограниченным бюджетом обучения. Поэтому полученные результаты прежде всего характеризуют сравнительное поведение моделей в данной постановке, а не их предельное качество после полноценной оптимизации.

2. **Зависимость от выбранного профиля данных.**  
   Основные результаты опираются на профиль `core_balanced`; при переходе к другой выборке структура классов и относительное положение моделей могут измениться.

**Источники:** `configs/forecasting_benchmark_smoke_v1.yaml`, `artifacts/manifests/forecasting_benchmark_smoke_v1_20260331T201416Z.json`, `artifacts/reports/meta_modeling_experiments_v1.xlsx`.


## 15. Воспроизводимость и навигация по репозиторию

Код проекта расположен в директории `src/`, конфигурации запусков — в `configs/`, а основные результаты экспериментов и отчёты — в `artifacts/`.

Для чтения результатов в первую очередь полезны следующие Excel-отчёты:
- `artifacts/reports/clustering_experiments_v1.xlsx`
- `artifacts/reports/cluster_profiles_v1.xlsx`
- `artifacts/reports/cluster_forecasting_analysis_v1.xlsx`
- `artifacts/reports/forecasting_smoke_summary_v1.xlsx`
- `artifacts/reports/meta_modeling_v1.xlsx`
- `artifacts/reports/meta_modeling_experiments_v1.xlsx`

Краткий маршрут воспроизведения ключевого результата выглядит так:

1. подготовить локальные пути к данным через `configs/paths.local.yaml`;
2. сформировать каталог рядов, профили и лог-доходности;
3. построить и отобрать финальные наборы признаков;
4. выполнить этап прогнозирования для разных моделей;
5. использовать результаты предыдущих этапов в задаче метамоделирования.

**Источники:** `README.md`, `PROJECT_STRUCTURE.md`, `DATA_LINEAGE.md`, `configs/*.yaml`, `artifacts/reports/*.xlsx`.

## 16. Окончательные выводы
1. Исследование подтверждает неуниверсальность моделей прогнозирования.
2. Анализ хаотической модели имеет смысл, но его следует интерпретировать с ограничениями по настройке.
3. Метамоделирование — последний практический механизм выбора модели в этом проекте.
4. Наибольший практический выигрыш в этом эксперименте наблюдается для direction_accuracy на горизонте 20.


## Рекомендуемые визуальные материалы
- `artifacts/figures/clusters/cluster_profile_heatmap_cfg_0448_with_chaos_agglomerative_quantile_pca_pca2_k8.png`
- `artifacts/figures/clusters/cluster_scatter_cfg_0448_with_chaos_agglomerative_quantile_pca_pca2_k8.png`
- `artifacts/figures/clusters/cluster_profile_heatmap_cfg_0185_base_gmm_quantile_pca_pca2_k4.png`


## Список файлов, используемых этим блокнотом
- `configs/*.yaml` (`forecasting_benchmark_smoke_v1.yaml`, `meta_modeling_experiments_v1.yaml`, `clustering_experiments_v1.yaml`, `cluster_profiling_v1.yaml`)
- `artifacts/processed/*.parquet`, `artifacts/features/*.parquet`, `artifacts/forecasting/*.parquet`
- `artifacts/clustering/*.csv`, `artifacts/analysis/*.csv`, `artifacts/meta_modeling/*.csv`
- `artifacts/reports/forecasting_smoke_summary_v1.xlsx`
- `artifacts/reports/clustering_experiments_v1.xlsx`
- `artifacts/reports/cluster_profiles_v1.xlsx`
- `artifacts/reports/cluster_forecasting_analysis_v1.xlsx`
- `artifacts/reports/meta_modeling_experiments_v1.xlsx`


## Элементы, требующие проверки вручную
1. Убедитесь, что все таблицы привязаны к запуску прогнозирования с series_count=300 и run_id прогнозирование_benchmark_smoke_v1_20260331T201416Z.
2. Подтвердите окончательную основную конфигурацию кластеризации для описания диссертации.
3. Добавьте итоговые цифры и проверьте читаемость русских надписей в Jupyter.
4. Перед замораживанием диссертации повторите проверку стабильности на более крупной выборке.
